# DINOv2 - improved: fine-tuning + richer features

## What changed from the original notebook

| Original | Improved |
|---|---|
| `dinov2-base` (768-dim) | `dinov2-large` (1024-dim) |
| Frozen extraction only | Two-stage: frozen extraction then fine-tuning of top blocks |
| CLS token only | CLS + mean of patch tokens (2048-dim) |
| 224x224 extraction | 336x336 |

### Why fine-tuning closes the gap with EfficientNetV2S

DINOv2 frozen features are excellent general-purpose representations,
but artist attribution is a specialised task. EfficientNetV2S was fine-tuned
on painting data. Fine-tuning the last few transformer blocks does the same for DINOv2.

### Workflow

1. Extract features at 336px using dinov2-large (CLS + patch mean, 2048-dim)
2. Train a Keras MLP head on frozen features - establishes a baseline
3. Fine-tune the last 4 transformer blocks end-to-end at a very low LR
4. Re-extract updated features from the fine-tuned backbone
5. Re-train the Keras head on the improved features


## 0 - Imports and config

In [1]:
import os, math
import numpy as np
from pathlib import Path
import tensorflow as tf
import keras
from keras import Model, layers
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, LearningRateScheduler
from keras.regularizers import l2
import tensorflow_addons as tfa

In [2]:
import torch
import torch.nn as nn
import torch.optim as torch_optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import time

DATA_DIR     = Path('../wikiart_split')
FEATURES_DIR = Path('./dino_features_v2')
CKPTS_DIR    = Path('./Checkpoints')
METRICS_DIR  = Path('./Metrics')

DINO_MODEL    = 'facebook/dinov2-large'
EXTRACT_SIZE  = 336    # used for frozen extraction (Step 1) and re-extraction (Step 4)
FINETUNE_SIZE = 224    # lower resolution for fine-tuning — 2.25x fewer pixels per image,
                       # major speed improvement; the backbone adapts equally well at 224
EXTRACT_BS    = 16
FINETUNE_BS   = 16     # increased from 8 — mixed precision frees enough VRAM to double this
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
N_UNFREEZE    = 4

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')
FEATURES_DIR.mkdir(exist_ok=True)


Device: cuda
GPU: NVIDIA GeForce GTX 1080  VRAM: 8.6 GB


## Step 1 - Extract richer features: CLS + patch mean

The CLS token summarises the image globally.
Patch tokens carry local spatial information - averaging them gives a
spatial summary that complements the global CLS representation.
Concatenating both (2048-dim for dinov2-large) is richer than CLS alone.


In [3]:
processor = AutoImageProcessor.from_pretrained(
    DINO_MODEL,
    size={'height': EXTRACT_SIZE, 'width': EXTRACT_SIZE},
)
dino = AutoModel.from_pretrained(DINO_MODEL).to(DEVICE)
dino.eval()

n_params = sum(p.numel() for p in dino.parameters())
print(f'Loaded {DINO_MODEL} ({n_params:,} params)')
print(f'Extraction resolution: {EXTRACT_SIZE}x{EXTRACT_SIZE}')


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

Loaded facebook/dinov2-large (304,368,640 params)
Extraction resolution: 336x336


In [4]:
class ArtDataset(Dataset):
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}

    def __init__(self, split_dir, class_names):
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples = []
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir(): continue
            idx = self.class_to_idx.get(class_dir.name)
            if idx is None: continue
            for img_path in sorted(class_dir.iterdir()):
                if img_path.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((img_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        return img, label, str(path)


def collate_fn(batch):
    images, labels, paths = zip(*batch)
    inputs = processor(images=list(images), return_tensors='pt')
    return inputs, torch.tensor(labels), paths


def extract_features(split, class_names, model=None, tag='v2'):
    feat_path  = FEATURES_DIR / f'{split}_features_{tag}.npy'
    label_path = FEATURES_DIR / f'{split}_labels_{tag}.npy'

    if feat_path.exists() and label_path.exists():
        print(f'  {split} ({tag}): loading from cache')
        return np.load(feat_path), np.load(label_path)

    backbone = model if model is not None else dino
    dataset  = ArtDataset(DATA_DIR / split, class_names)
    loader   = DataLoader(dataset, batch_size=EXTRACT_BS, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

    all_feats, all_labels = [], []
    print(f'  Extracting {split} ({len(dataset)} images, tag={tag})...')

    backbone.eval()
    with torch.no_grad():
        for i, (inputs, labels, _) in enumerate(loader):
            inputs     = {k: v.to(DEVICE) for k, v in inputs.items()}
            out        = backbone(**inputs)
            cls        = out.last_hidden_state[:, 0, :]
            patch_mean = out.last_hidden_state[:, 1:, :].mean(dim=1)
            feats      = torch.cat([cls, patch_mean], dim=1)
            all_feats.append(feats.cpu().numpy())
            all_labels.append(labels.numpy())
            if (i + 1) % 10 == 0:
                print(f'    {min((i+1)*EXTRACT_BS, len(dataset))}/{len(dataset)}')

    feats  = np.concatenate(all_feats,  axis=0).astype(np.float32)
    labels = np.concatenate(all_labels, axis=0).astype(np.int32)
    np.save(feat_path, feats)
    np.save(label_path, labels)
    print(f'  Saved shape={feats.shape}')
    return feats, labels


In [5]:
_tmp = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', batch_size=None, image_size=(64, 64))
class_names = _tmp.class_names
N_CLASSES   = len(class_names)
print(f'Classes ({N_CLASSES}): {class_names}')
del _tmp

print('Extracting frozen DINOv2-large features (CLS + patch mean, 336px)...')
train_feats, train_labels = extract_features('train', class_names, tag='frozen')
val_feats,   val_labels   = extract_features('val',   class_names, tag='frozen')
test_feats,  test_labels  = extract_features('test',  class_names, tag='frozen')

FEAT_DIM = train_feats.shape[1]
print(f'Feature dimension: {FEAT_DIM}  (expected 2048 for dinov2-large)')


Found 9326 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']
Extracting frozen DINOv2-large features (CLS + patch mean, 336px)...
  train (frozen): loading from cache
  val (frozen): loading from cache
  test (frozen): loading from cache
Feature dimension: 2048  (expected 2048 for dinov2-large)


## Step 2 - Train Keras head on frozen features

Establishes a strong baseline before the more expensive fine-tuning step.

In [6]:
KERAS_BATCH = 256
AUTOTUNE    = tf.data.AUTOTUNE

def make_tf_datasets(train_f, train_l, val_f, val_l, test_f, test_l, n_classes):
    oh = lambda l: tf.one_hot(l, n_classes).numpy()
    train_ds = (tf.data.Dataset.from_tensor_slices((train_f, oh(train_l)))
                  .shuffle(len(train_f), seed=123).batch(KERAS_BATCH).prefetch(AUTOTUNE))
    val_ds   = (tf.data.Dataset.from_tensor_slices((val_f, oh(val_l)))
                  .batch(KERAS_BATCH).prefetch(AUTOTUNE))
    test_ds  = (tf.data.Dataset.from_tensor_slices((test_f, oh(test_l)))
                  .batch(KERAS_BATCH).prefetch(AUTOTUNE))
    return train_ds, val_ds, test_ds

train_tf, val_tf, test_tf = make_tf_datasets(
    train_feats, train_labels, val_feats, val_labels, test_feats, test_labels, N_CLASSES)


In [7]:
def build_dino_classifier(input_dim, n_classes=23, dropout=0.4, l2_reg=1e-4, name='dino_classifier'):
    inp = keras.Input(shape=(input_dim,), name='features')
    x   = layers.LayerNormalization(name='input_norm')(inp)
    x   = layers.Dense(1024, kernel_regularizer=l2(l2_reg), name='fc1')(x)
    x   = layers.BatchNormalization(name='bn1')(x)
    x   = layers.Activation('gelu', name='act1')(x)
    x   = layers.Dropout(dropout, name='drop1')(x)
    x   = layers.Dense(512, kernel_regularizer=l2(l2_reg), name='fc2')(x)
    x   = layers.BatchNormalization(name='bn2')(x)
    x   = layers.Activation('gelu', name='act2')(x)
    x   = layers.Dropout(dropout, name='drop2')(x)
    x   = layers.Dense(256, kernel_regularizer=l2(l2_reg), name='fc3')(x)
    x   = layers.BatchNormalization(name='bn3')(x)
    x   = layers.Activation('gelu', name='act3')(x)
    x   = layers.Dropout(dropout / 2, name='drop3')(x)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32', name='head')(x)
    return Model(inp, out, name=name)


def cosine_warmup(base_lr, total_epochs, warmup_epochs=5):
    def sched(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))
    return sched


def train_keras_head(train_ds, val_ds, feat_dim, n_classes,
                     epochs=60, lr=3e-4, ckpt_name='ckpt_dino', log_name='log_dino.csv'):
    model = build_dino_classifier(feat_dim, n_classes)
    model.compile(
        optimizer=tfa.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
        loss=CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[CategoricalAccuracy(name='accuracy'),
                 AUC(multi_label=True, name='auc'),
                 tfa.metrics.F1Score(num_classes= n_classes, average='macro', name='f1_score')],
    )
    cbs = [
        ModelCheckpoint(f'{ckpt_name}.keras', monitor='val_loss', save_best_only=True, verbose=1),
        CSVLogger(log_name),
        LearningRateScheduler(cosine_warmup(lr, epochs, warmup_epochs=5)),
        EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=1)
    return model


print('Training Keras head on frozen DINOv2-large features...')
model_dino_frozen = train_keras_head(
    train_tf, val_tf, FEAT_DIM, N_CLASSES,
    ckpt_name=CKPTS_DIR / 'ckpt_dino_frozen', log_name= METRICS_DIR / 'log_dino_frozen.csv'
)

frozen_results = model_dino_frozen.evaluate(test_tf, return_dict=True, verbose=0)
print('Frozen backbone - test results:')
for k, v in frozen_results.items():
    print(f'  {k}: {v:.4f}')


Training Keras head on frozen DINOv2-large features...
Epoch 1/60
35/37 [===========================>..] - ETA: 0s - loss: 3.4404 - accuracy: 0.1080 - auc: 0.6229 - f1_score: 0.0939
Epoch 1: val_loss improved from inf to 2.82646, saving model to Checkpoints\ckpt_dino_frozen.keras
37/37 [==============================] - 6s 53ms/step - loss: 3.4240 - accuracy: 0.1119 - auc: 0.6275 - f1_score: 0.0973 - val_loss: 2.8265 - val_accuracy: 0.3449 - val_auc: 0.8204 - val_f1_score: 0.2832 - lr: 6.0000e-05
Epoch 2/60
35/37 [===========================>..] - ETA: 0s - loss: 2.6816 - accuracy: 0.3494 - auc: 0.8203 - f1_score: 0.3005
Epoch 2: val_loss improved from 2.82646 to 2.27379, saving model to Checkpoints\ckpt_dino_frozen.keras
37/37 [==============================] - 1s 30ms/step - loss: 2.6710 - accuracy: 0.3533 - auc: 0.8229 - f1_score: 0.3042 - val_loss: 2.2738 - val_accuracy: 0.5115 - val_auc: 0.9066 - val_f1_score: 0.4314 - lr: 1.2000e-04
Epoch 3/60
36/37 [============================>

## Step 3 - Fine-tune dinov2-base (with VRAM cleanup)

### The real cause of the slowdown

After Step 1-2 completed, the dinov2-large model (~1.3 GB in VRAM), all of its
frozen extraction intermediate buffers, and the Keras head model were still sitting
in GPU memory. When Cell 12 then loaded dinov2-base alongside them, and Cell 15's
`copy.deepcopy` created a THIRD backbone, available VRAM dropped below what's
needed for backprop activations. CUDA silently spilled to system RAM over PCIe,
which runs ~27x slower than VRAM — explaining the 858s/batch and the system stutter.

### The fixes applied below

1. **Free VRAM from Steps 1-2 first** (new Cell 11.5): delete dinov2-large,
   teacher Keras model, extracted tensors, run garbage collection, empty cache.
2. **Avoid the deepcopy**: load a fresh dinov2-base directly inside finetune_backbone
   instead of deepcopying a pre-loaded one. This prevents a period where two full
   copies exist in VRAM simultaneously.
3. **Enable gradient checkpointing**: trades a ~20% compute increase for ~40%
   activation memory reduction. Critical for 8GB VRAM with a transformer backbone.
4. **Smaller batch size (8)**: activation memory scales linearly with batch size.
   Combined with gradient accumulation (accumulate 2 batches -> effective batch 16),
   this gives the same optimization dynamics with half the peak memory.


In [8]:
# ── Free VRAM before starting Step 3 ──────────────────────────────────────
# Step 1-2 left dinov2-large, Keras teacher, and feature tensors in VRAM.
# Fine-tuning needs the maximum possible free VRAM for backprop activations.
#
# IMPORTANT: if you still see slow batch times after this, restart the Jupyter
# kernel (Kernel > Restart) and run only cells 2, 3, 6, 7 (for class_names),
# cell 12 onwards. Kernel restart is the only 100%-reliable way to fully
# release VRAM that HuggingFace/PyTorch has stubbornly cached.
import gc

# Null out references first
for name in ('dino', 'model_dino_frozen', 'train_tf', 'val_tf', 'test_tf',
             'train_feats', 'val_feats', 'test_feats',
             'train_labels', 'val_labels', 'test_labels'):
    if name in globals():
        try:
            exec(f'del {name}')
            print(f'  Deleted: {name}')
        except NameError:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'\nGPU memory after cleanup: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')
    if free < 6e9:
        print('WARNING: less than 6 GB free — restart the kernel and run cells')
        print('         2, 3, 6 (for class_names), then 12 onwards to reset.')
    else:
        print('OK — enough free VRAM to proceed.')


  Deleted: dino
  Deleted: model_dino_frozen
  Deleted: train_tf
  Deleted: val_tf
  Deleted: test_tf
  Deleted: train_feats
  Deleted: val_feats
  Deleted: test_feats
  Deleted: train_labels
  Deleted: val_labels
  Deleted: test_labels

GPU memory after cleanup: 1.54 GB free / 8.59 GB total
         2, 3, 6 (for class_names), then 12 onwards to reset.


In [9]:
# We will NOT pre-load dinov2-base here. Loading it inside finetune_backbone()
# ensures it's the only backbone in VRAM at any point in time (no deepcopy overhead).
DINO_BASE_MODEL = 'facebook/dinov2-base'
FINETUNE_SIZE   = 224

finetune_processor = AutoImageProcessor.from_pretrained(
    DINO_BASE_MODEL,
    size={'height': FINETUNE_SIZE, 'width': FINETUNE_SIZE},
)


class ArtDatasetFull(Dataset):
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}

    def __init__(self, split_dir, class_names):
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples = []
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir(): continue
            idx = self.class_to_idx.get(class_dir.name)
            if idx is None: continue
            for img_path in sorted(class_dir.iterdir()):
                if img_path.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((img_path, idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return Image.open(path).convert('RGB'), label


def collate_full(batch):
    images, labels = zip(*batch)
    inputs = finetune_processor(images=list(images), return_tensors='pt')
    return inputs, torch.tensor(labels, dtype=torch.long)


In [10]:
class DinoWithHead(nn.Module):
    """DINOv2 backbone + linear head for end-to-end fine-tuning.
    Only used during the fine-tuning phase — afterwards the backbone
    is detached and used for feature re-extraction."""
    def __init__(self, backbone, feat_dim, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(feat_dim, n_classes)

    def forward(self, inputs):
        out        = self.backbone(**inputs)
        cls        = out.last_hidden_state[:, 0, :]
        patch_mean = out.last_hidden_state[:, 1:, :].mean(dim=1)
        return self.head(torch.cat([cls, patch_mean], dim=1))

In [11]:
def unfreeze_top_blocks(model, n_blocks):
    for p in model.parameters():
        p.requires_grad = False
    total_blocks = len(model.encoder.layer)
    for block in model.encoder.layer[total_blocks - n_blocks:]:
        for p in block.parameters():
            p.requires_grad = True
    if hasattr(model, 'layernorm'):
        for p in model.layernorm.parameters():
            p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Unfroze last {n_blocks}/{total_blocks} blocks: {trainable:,}/{total:,} params trainable')


def finetune_backbone(model_name, class_names, n_unfreeze=2,
                      epochs=10, lr=5e-6, warmup_epochs=2,
                      batch_size=8, grad_accum_steps=2):
    """
    Fine-tune a DINOv2 backbone end-to-end.

    Key VRAM-saving techniques:
    - Fresh model loaded inside this function (no deepcopy = no transient 2x VRAM)
    - gradient_checkpointing_enable(): trades compute for memory by recomputing
      activations during backward pass instead of storing them. ~40% memory savings
      on transformer backbones at ~20% speed cost. Essential for 8GB VRAM.
    - Smaller batch_size=8 with grad_accum_steps=2 gives effective batch=16 with
      only half the peak activation memory.
    """
    # Load fresh — nothing else of this backbone in VRAM simultaneously
    backbone = AutoModel.from_pretrained(model_name).to(DEVICE)
    unfreeze_top_blocks(backbone, n_unfreeze)

    # Enable gradient checkpointing on the backbone.
    # HuggingFace transformers support this via a single method call.
    if hasattr(backbone, 'gradient_checkpointing_enable'):
        backbone.gradient_checkpointing_enable()
        print('Gradient checkpointing enabled')

    feat_dim   = 2 * backbone.config.hidden_size
    full_model = DinoWithHead(backbone, feat_dim, len(class_names)).to(DEVICE)

    train_loader = DataLoader(ArtDatasetFull(DATA_DIR / 'train', class_names),
                              batch_size=batch_size, shuffle=True,
                              collate_fn=collate_full, num_workers=0)
    val_loader   = DataLoader(ArtDatasetFull(DATA_DIR / 'val', class_names),
                              batch_size=batch_size, shuffle=False,
                              collate_fn=collate_full, num_workers=0)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch_optim.AdamW(
        filter(lambda p: p.requires_grad, full_model.parameters()),
        lr=lr, weight_decay=1e-5)

    # Scheduler counts OPTIMIZER STEPS, not forward passes
    steps_per_epoch = len(train_loader) // grad_accum_steps
    warmup_steps    = warmup_epochs * steps_per_epoch
    scheduler       = torch_optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min(1.0, step / max(1, warmup_steps)))

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    best_val_loss = float('inf')
    best_state    = None
    n_batches     = len(train_loader)

    print(f'\nConfiguration:')
    print(f'  Model: {model_name}')
    print(f'  Resolution: {FINETUNE_SIZE}x{FINETUNE_SIZE}')
    print(f'  Batch size: {batch_size}  grad accum: {grad_accum_steps}  effective: {batch_size*grad_accum_steps}')
    print(f'  Epochs: {epochs}  batches/epoch: {n_batches}')
    print(f'  Gradient checkpointing: on')
    print(f'  Mixed precision: on')

    # Timing batch
    print('\nRunning a timing batch...')
    full_model.train()
    t0 = time.time()
    sample_inputs, sample_labels = next(iter(train_loader))
    sample_inputs = {k: v.to(DEVICE) for k, v in sample_inputs.items()}
    sample_labels = sample_labels.to(DEVICE)
    with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
        loss = criterion(full_model(sample_inputs), sample_labels)
    scaler.scale(loss).backward()
    optimizer.zero_grad()
    torch.cuda.synchronize()
    secs_per_batch = time.time() - t0
    est_hours = secs_per_batch * n_batches * epochs / 3600
    print(f'~{secs_per_batch:.2f}s/batch -> estimated total: {est_hours:.2f} hours\n')

    if secs_per_batch > 30:
        print('WARNING: batch time >30s indicates VRAM is spilling to RAM.')
        print('Consider restarting the kernel before running this cell.')
        print('Continuing anyway...\n')

    # ── Training ──────────────────────────────────────────────────────────────
    for epoch in range(epochs):
        full_model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        epoch_start = time.time()
        optimizer.zero_grad()

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            labels = labels.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = full_model(inputs)
                loss   = criterion(logits, labels) / grad_accum_steps

            scaler.scale(loss).backward()

            # Only step the optimizer every grad_accum_steps batches
            if (batch_idx + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(full_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            t_loss    += loss.item() * labels.size(0) * grad_accum_steps
            t_correct += (logits.detach().argmax(1) == labels).sum().item()
            t_total   += labels.size(0)

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == n_batches:
                elapsed = time.time() - epoch_start
                pct     = (batch_idx + 1) / n_batches
                eta     = elapsed / pct * (1 - pct)
                print(f'  [{epoch+1}/{epochs}] batch {batch_idx+1}/{n_batches}  '
                      f'loss={t_loss/t_total:.4f}  '
                      f'elapsed={elapsed:.0f}s  eta={eta:.0f}s', end='\r')

        print()

        # ── Validation ────────────────────────────────────────────────────────
        full_model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
                labels = labels.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                    logits = full_model(inputs)
                v_loss    += criterion(logits, labels).item() * labels.size(0)
                v_correct += (logits.argmax(1) == labels).sum().item()
                v_total   += labels.size(0)

        tl = t_loss / t_total
        vl = v_loss / v_total
        epoch_mins = (time.time() - epoch_start) / 60
        print(f'Epoch {epoch+1:02d}/{epochs}  '
              f'train_loss={tl:.4f}  train_acc={t_correct/t_total:.4f}  '
              f'val_loss={vl:.4f}  val_acc={v_correct/v_total:.4f}  '
              f'({epoch_mins:.1f} min)')

        if vl < best_val_loss:
            best_val_loss = vl
            best_state    = {k: v.cpu().clone()
                             for k, v in full_model.backbone.state_dict().items()}
            print(f'  New best val_loss: {best_val_loss:.4f}')

    full_model.backbone.load_state_dict(best_state)
    # Disable gradient checkpointing before returning — re-extraction doesn't need it
    if hasattr(full_model.backbone, 'gradient_checkpointing_disable'):
        full_model.backbone.gradient_checkpointing_disable()
    print(f'\nFine-tuning complete. Best val_loss: {best_val_loss:.4f}')
    return full_model.backbone


In [12]:
# Fine-tune dinov2-base with 2 blocks unfrozen.
# 2/12 blocks is proportionally equivalent to 4/24 in dinov2-large.
N_UNFREEZE_BASE = 2

print(f'Fine-tuning {DINO_BASE_MODEL} with gradient checkpointing...')
dino_finetuned = finetune_backbone(
    DINO_BASE_MODEL,         # name, not model — loaded fresh inside the function
    class_names,
    n_unfreeze=N_UNFREEZE_BASE,
    epochs=10,
    lr=5e-6,
    warmup_epochs=2,
    batch_size=8,
    grad_accum_steps=2,
)


Fine-tuning facebook/dinov2-base with gradient checkpointing...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Unfroze last 2/12 blocks: 14,180,352/86,580,480 params trainable
Gradient checkpointing enabled

Configuration:
  Model: facebook/dinov2-base
  Resolution: 224x224
  Batch size: 8  grad accum: 2  effective: 16
  Epochs: 10  batches/epoch: 1166
  Gradient checkpointing: on
  Mixed precision: on

Running a timing batch...


C:\Users\franc\AppData\Local\Temp\ipykernel_18236\3599557352.py:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
C:\Users\franc\AppData\Local\Temp\ipykernel_18236\3599557352.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):


~1.06s/batch -> estimated total: 3.43 hours



C:\Users\franc\AppData\Local\Temp\ipykernel_18236\3599557352.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
c:\Users\franc\miniconda3\envs\keras-gpu\lib\site-packages\torch\optim\lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  [1/10] batch 1166/1166  loss=2.7951  elapsed=348s  eta=0ss


C:\Users\franc\AppData\Local\Temp\ipykernel_18236\3599557352.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):


Epoch 01/10  train_loss=2.7951  train_acc=0.2246  val_loss=2.1288  val_acc=0.4769  (6.7 min)
  New best val_loss: 2.1288
  [2/10] batch 1166/1166  loss=1.6621  elapsed=214s  eta=0ss
Epoch 02/10  train_loss=1.6621  train_acc=0.6405  val_loss=1.4355  val_acc=0.7149  (4.1 min)
  New best val_loss: 1.4355
  [3/10] batch 1166/1166  loss=1.2036  elapsed=208s  eta=0ss
Epoch 03/10  train_loss=1.2036  train_acc=0.8057  val_loss=1.2457  val_acc=0.7887  (4.0 min)
  New best val_loss: 1.2457
  [4/10] batch 1166/1166  loss=0.9893  elapsed=210s  eta=0ss
Epoch 04/10  train_loss=0.9893  train_acc=0.9003  val_loss=1.1732  val_acc=0.8082  (4.0 min)
  New best val_loss: 1.1732
  [5/10] batch 1166/1166  loss=0.8623  elapsed=218s  eta=0ss
Epoch 05/10  train_loss=0.8623  train_acc=0.9522  val_loss=1.0941  val_acc=0.8444  (4.2 min)
  New best val_loss: 1.0941
  [6/10] batch 1166/1166  loss=0.7736  elapsed=219s  eta=0ss
Epoch 06/10  train_loss=0.7736  train_acc=0.9814  val_loss=1.0777  val_acc=0.8494  (4.2 mi

## Step 4 - Re-extract features and retrain Keras head

In [13]:
# Re-extract features from the fine-tuned dinov2-base backbone.
# Note: extraction still uses the 336px processor (the original `processor` variable)
# to maintain consistent resolution with the frozen dinov2-large features.
# The fine-tuned backbone handles variable resolution via position embedding interpolation.
print('Re-extracting features from fine-tuned dinov2-base backbone (at 336px)...')
train_feats_ft, train_labels_ft = extract_features('train', class_names, model=dino_finetuned, tag='base_finetuned')
val_feats_ft,   val_labels_ft   = extract_features('val',   class_names, model=dino_finetuned, tag='base_finetuned')
test_feats_ft,  test_labels_ft  = extract_features('test',  class_names, model=dino_finetuned, tag='base_finetuned')

FEAT_DIM_FT = train_feats_ft.shape[1]
print(f'Fine-tuned feature dimension: {FEAT_DIM_FT}  (expected 1536 for dinov2-base)')

train_tf_ft, val_tf_ft, test_tf_ft = make_tf_datasets(
    train_feats_ft, train_labels_ft,
    val_feats_ft,   val_labels_ft,
    test_feats_ft,  test_labels_ft,
    N_CLASSES
)


Re-extracting features from fine-tuned dinov2-base backbone (at 336px)...
  Extracting train (9326 images, tag=base_finetuned)...
    160/9326
    320/9326
    480/9326
    640/9326
    800/9326
    960/9326
    1120/9326
    1280/9326
    1440/9326
    1600/9326
    1760/9326
    1920/9326
    2080/9326
    2240/9326
    2400/9326
    2560/9326
    2720/9326
    2880/9326
    3040/9326
    3200/9326
    3360/9326
    3520/9326
    3680/9326
    3840/9326
    4000/9326
    4160/9326
    4320/9326
    4480/9326
    4640/9326
    4800/9326
    4960/9326
    5120/9326
    5280/9326
    5440/9326
    5600/9326
    5760/9326
    5920/9326
    6080/9326
    6240/9326
    6400/9326
    6560/9326
    6720/9326
    6880/9326
    7040/9326
    7200/9326
    7360/9326
    7520/9326
    7680/9326
    7840/9326
    8000/9326
    8160/9326
    8320/9326
    8480/9326
    8640/9326
    8800/9326
    8960/9326
    9120/9326
    9280/9326
  Saved shape=(9326, 1536)
  Extracting val (1992 images, tag=ba

In [14]:
print('Training Keras head on fine-tuned dinov2-base features...')
model_dino_ft = train_keras_head(
    train_tf_ft, val_tf_ft,
    FEAT_DIM_FT,   # 1536 for dinov2-base (was 2048 for dinov2-large)
    N_CLASSES,
    ckpt_name=CKPTS_DIR / 'ckpt_dino_base_finetuned',
    log_name=METRICS_DIR / 'log_dino_base_finetuned.csv'
)

ft_results = model_dino_ft.evaluate(test_tf_ft, return_dict=True, verbose=0)
print('Fine-tuned dinov2-base - test results:')
for k, v in ft_results.items():
    print(f'  {k}: {v:.4f}')

print('=' * 60)
print('Comparison: frozen dinov2-large vs fine-tuned dinov2-base')
print('=' * 60)
print(f"{'Metric':<15} {'Frozen-large':>14} {'FT-base':>10} {'Delta':>8}")
print('-' * 52)
for k in frozen_results:
    if k not in ft_results:
        continue
    delta = ft_results[k] - frozen_results[k]
    sign  = '+' if delta >= 0 else ''
    print(f'{k:<15} {frozen_results[k]:>14.4f} {ft_results[k]:>10.4f} {sign}{delta:>7.4f}')
print()
print('Note: frozen uses dinov2-large (2048-dim), fine-tuned uses dinov2-base (1536-dim).')
print('The fine-tuned model has domain-adapted representations; the frozen model has')
print('richer general-purpose features. Both comparisons are valid and interesting.')


Training Keras head on fine-tuned dinov2-base features...
Epoch 1/60
36/37 [============================>.] - ETA: 0s - loss: 3.1466 - accuracy: 0.2051 - auc: 0.6873 - f1_score: 0.1718
Epoch 1: val_loss improved from inf to 2.40761, saving model to Checkpoints\ckpt_dino_base_finetuned.keras
37/37 [==============================] - 3s 33ms/step - loss: 3.1411 - accuracy: 0.2068 - auc: 0.6886 - f1_score: 0.1732 - val_loss: 2.4076 - val_accuracy: 0.5532 - val_auc: 0.9023 - val_f1_score: 0.4649 - lr: 6.0000e-05
Epoch 2/60
37/37 [==============================] - ETA: 0s - loss: 2.1509 - accuracy: 0.5652 - auc: 0.9066 - f1_score: 0.4874
Epoch 2: val_loss improved from 2.40761 to 1.80490, saving model to Checkpoints\ckpt_dino_base_finetuned.keras
37/37 [==============================] - 1s 29ms/step - loss: 2.1509 - accuracy: 0.5652 - auc: 0.9066 - f1_score: 0.4874 - val_loss: 1.8049 - val_accuracy: 0.7033 - val_auc: 0.9604 - val_f1_score: 0.6372 - lr: 1.2000e-04
Epoch 3/60
35/37 [==========